In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import mean_squared_error,r2_score
from micromlgen import port

In [2]:
data = pd.read_csv('D:\data_cleaned.txt',header=None)
pd.set_option('display.expand_frame_repr', False)
# print(data.head(20))

In [3]:
data.rename(columns={data.columns[-1]: 'Label'}, inplace=True)
before = len(data)
data = data[data['Label'] != 255]
after = len(data)
# print(f"删除前：{before} 行，删除后：{after} 行，共删除了 {before - after} 行。")
# print(data['Label'].value_counts().sort_index())
print(f" {data['Label'].value_counts().sort_index()}")


 Label
0       31
1       41
2      162
3      383
4      738
5      993
6     1073
7     1273
8     1548
9     1556
10     943
11     290
12      62
13      28
14       8
15       2
Name: count, dtype: int64


In [4]:
X = data.iloc[:, :-1]
y = data.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# ===========================
# 新增：F-score 可视化
# ===========================
# import matplotlib.pyplot as plt

# def plot_f_scores(X, y, k_best=None, top_n=None, figsize=(12,6)):
#     """
#     可视化每个特征的 ANOVA F-score，从高到低排序。
#     """
#     # 1. 特征名
#     if hasattr(X, "columns"):
#         feature_names = list(X.columns)
#     else:
#         feature_names = [f"f{i}" for i in range(X.shape[1])]

#     # 2. 获取 F-scores
#     if k_best is not None and hasattr(k_best, "scores_"):
#         scores = np.nan_to_num(k_best.scores_, nan=0.0)
#     else:
#         scores, _ = f_classif(X, y)
#         scores = np.nan_to_num(scores, nan=0.0)

#     # 3. 排序
#     idx_sorted = np.argsort(scores)[::-1]
#     names_sorted = [feature_names[i] for i in idx_sorted]
#     scores_sorted = scores[idx_sorted]

#     # 若指定 top_n，则截断
#     if top_n is not None:
#         names_sorted = names_sorted[:top_n]
#         scores_sorted = scores_sorted[:top_n]

#     # 4. 绘图
#     plt.figure(figsize=figsize)
#     x = np.arange(len(names_sorted))
#     plt.bar(x, scores_sorted)
#     plt.xticks(x, names_sorted, rotation=90)
#     plt.xlabel("Feature")
#     plt.ylabel("ANOVA F-score")
#     plt.title("Feature Importance (F-score)")
#     plt.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.6)
#     plt.tight_layout()
#     plt.show()


# # 调用可视化函数：显示前 40 个最重要特征（可调整）
# plot_f_scores(X_train, y_train, k_best=k_best, top_n=30)

In [5]:
import warnings
warnings.filterwarnings("ignore")

k = 12 #16 21 24 28
k_best = SelectKBest(score_func=f_classif, k=k)
k_best.fit(X_train, y_train)

selected_feature_indices = k_best.get_support(indices=True)
# we have to print it like this to have the commas between the indices so that it's easy to copy and paste to Arduino IDE
print("selected features: ", X.columns[selected_feature_indices])


selected features:  Index([166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177], dtype='object')


In [ ]:
# # # # anticlockwise k=12 max_depth=3 n_estimators=50
# # # # clockwise     k=10 max_depth=4 

clf = RandomForestClassifier(
    random_state=42
    )

param_grid = {
    "n_estimators": [88,89,90,91,92],
    "max_depth":[3]
}

grid = GridSearchCV(
    clf,
    param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)
print(grid.best_params_)

{'max_depth': 3, 'n_estimators': 88}


: 

In [ ]:
clf = RandomForestClassifier(
    n_estimators=50,
    max_depth=3, 
    random_state=42)

clf.fit(X_train.iloc[:, selected_feature_indices], y_train)
y_pred = clf.predict(X_test.iloc[:, selected_feature_indices])

accuracy1 = accuracy_score(y_test, y_pred)
print(f'Accuracy1: {accuracy1}')

def tolerant_accuracy(y_true, y_pred, tol=1):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean(np.abs(y_pred - y_true) <= tol)

acc_tol1 = tolerant_accuracy(y_test, y_pred, tol=2)
print(f"Tolerance-1 accuracy: {acc_tol1:.4f}")


Accuracy1: 0.1954954954954955
Tolerance-1 accuracy: 0.7225


In [ ]:
# clf2 = AdaBoostRegressor(
#     n_estimators=51,
#     learning_rate=0.5,
#     random_state=42)
# clf2.fit(X_train.iloc[:, selected_feature_indices], y_train)
# y_pred2 = clf2.predict(X_test.iloc[:, selected_feature_indices])
# accuracy2 = r2_score(y_test, y_pred2)
# print(f'Accuracy2: {accuracy2}')

In [ ]:
arduino_code = open("randomForest.h", mode="w")
arduino_code.write(port(clf))
arduino_code.close()
print("selected features: ", X.columns[selected_feature_indices])

selected features:  Index([22, 23, 24, 25, 26, 27, 28, 29], dtype='object')
